In [6]:
#Cell 1 — retrain a smaller 7-feature model on CICIDS2017:
import sys, os, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import joblib

from src.utils import load_config, set_seed
from src.models.advanced import train_random_forest
from src.evaluation.metrics import evaluate_model, print_metrics
from src.cross_dataset.alignment import (
    FEATURE_NAME_MAP_2017_TO_UNSW, build_alignment_report,
    print_alignment_report, rename_target_to_source_schema
)
from src.preprocessing.cross_dataset_cleaning import clean_target_dataset

cfg = load_config()
set_seed(cfg["project"]["random_state"])
seed = cfg["project"]["random_state"]
processed_dir = cfg["paths"]["processed_dir"]
models_dir = cfg["paths"]["models_dir"]

# Load full raw (unscaled) 2017 train/test data
X_train_raw = pd.read_parquet(os.path.join(processed_dir, "X_train_raw.parquet"))
X_test_raw = pd.read_parquet(os.path.join(processed_dir, "X_test_raw.parquet"))
y_train = pd.read_parquet(os.path.join(processed_dir, "y_train.parquet"))["label_binary"]
y_test = pd.read_parquet(os.path.join(processed_dir, "y_test.parquet"))["label_binary"]

# Common features (the 7 that exist in both 2017 and UNSW-NB15)
common_features_2017_names = list(FEATURE_NAME_MAP_2017_TO_UNSW.keys())
print("Common features (2017 names):", common_features_2017_names)

X_train_common = X_train_raw[common_features_2017_names]
X_test_common = X_test_raw[common_features_2017_names]

# Retrain RF using ONLY these 7 features
rf_7feat, train_time = train_random_forest(X_train_common, y_train, random_state=seed)

in_dataset_metrics_7feat = evaluate_model(rf_7feat, X_test_common, y_test, model_name="RF - 7 common features (2017 in-dataset)")
in_dataset_metrics_7feat["train_time_sec"] = round(train_time, 4)

print_metrics(in_dataset_metrics_7feat)

joblib.dump(rf_7feat, os.path.join(models_dir, "rf_7feature_for_unsw.joblib"))

Common features (2017 names): ['Total Fwd Packets', 'Fwd Packet Length Mean', 'Total Length of Fwd Packets', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'Fwd IAT Mean', 'Bwd IAT Mean']
RESULTS: RF - 7 common features (2017 in-dataset)
Accuracy   : 0.9999
Precision  : 0.9998
Recall     : 0.9999
F1-score   : 0.9999
ROC-AUC    : 1.0000
PR-AUC     : 1.0000
FPR        : 0.0002
FNR        : 0.0001
------------------------------------------------------------
Confusion Matrix:
              Predicted BENIGN   Predicted DDoS
Actual BENIGN        19,009                 4
Actual DDoS               2            25,601
------------------------------------------------------------
Training time    : 9.5177 sec
Prediction time  : 0.2886 sec


['d:\\MSc_Project\\Network_IDS_Framework\\results/models\\rf_7feature_for_unsw.joblib']

In [7]:
#Now — align and test on UNSW-NB15
#Add Cell 2:
# --- Build alignment report for UNSW-NB15 ---
unsw_alignment_report = build_alignment_report(
    common_features_2017_names,
    mapping=FEATURE_NAME_MAP_2017_TO_UNSW
)
print_alignment_report(unsw_alignment_report)

# Save the report for the thesis appendix
with open(os.path.join(cfg["paths"]["reports_dir"], "unsw_alignment_report.json"), "w") as f:
    json.dump(unsw_alignment_report, f, indent=2)

CROSS-DATASET FEATURE ALIGNMENT REPORT
Total source (2017) features:     7
Successfully mapped to target:    7
Unmapped (dropped):               0
Final aligned feature count:      7
----------------------------------------------------------------
Mapping used (2017 -> target):
    Total Fwd Packets                   -> spkts
    Fwd Packet Length Mean              -> smean
    Total Length of Fwd Packets         -> sbytes
    Init_Win_bytes_forward              -> swin
    Init_Win_bytes_backward             -> dwin
    Fwd IAT Mean                        -> sinpkt
    Bwd IAT Mean                        -> dinpkt


In [8]:
#Cell 3 — load, align, and clean UNSW-NB15:
unsw_dir = os.path.join(cfg["project_root"], "datasets", "UNSW_NB15")
df_unsw_train = pd.read_csv(os.path.join(unsw_dir, "UNSW_NB15_training-set.csv"), low_memory=False)
df_unsw_test = pd.read_csv(os.path.join(unsw_dir, "UNSW_NB15_testing-set.csv"), low_memory=False)
df_unsw = pd.concat([df_unsw_train, df_unsw_test], ignore_index=True)
df_unsw.columns = [str(c).strip() for c in df_unsw.columns]

X_unsw_aligned = rename_target_to_source_schema(df_unsw, unsw_alignment_report)
y_unsw_raw = df_unsw["label"]  # already binary: 0 = normal, 1 = attack

print("Aligned UNSW-NB15 shape:", X_unsw_aligned.shape)
print(X_unsw_aligned.head(3))

# Clean (dedupe, remove inf/NaN) - reusing the same target-cleaning function from Step 11
X_unsw_clean, y_unsw_clean = clean_target_dataset(X_unsw_aligned, y_unsw_raw)

print("\nFinal cleaned UNSW-NB15 shape:", X_unsw_clean.shape)
print("Label distribution:")
print(y_unsw_clean.value_counts())

Aligned UNSW-NB15 shape: (257673, 7)
   Total Fwd Packets  Fwd Packet Length Mean  Total Length of Fwd Packets  \
0                  2                     248                          496   
1                  2                     881                         1762   
2                  2                     534                         1068   

   Init_Win_bytes_forward  Init_Win_bytes_backward  Fwd IAT Mean  Bwd IAT Mean  
0                       0                        0         0.011           0.0  
1                       0                        0         0.008           0.0  
2                       0                        0         0.005           0.0  
TARGET DATASET (2018) CLEANING REPORT
Rows at start:                257,673
Duplicate rows removed:       131,288
Infinite values found:        0
Rows removed (NaN/inf):       0
Final rows:                   126,385

Final cleaned UNSW-NB15 shape: (126385, 7)
Label distribution:
_label_raw
0    70668
1    55717
Name: count, dtyp

In [9]:
#Cell 4 — run the cross-dataset evaluation:
cross_metrics_unsw = evaluate_model(rf_7feat, X_unsw_clean, y_unsw_clean, model_name="RF (7-feat) - Cross-Dataset (UNSW-NB15)")
print_metrics(cross_metrics_unsw)

RESULTS: RF (7-feat) - Cross-Dataset (UNSW-NB15)
Accuracy   : 0.5591
Precision  : 0.0000
Recall     : 0.0000
F1-score   : 0.0000
ROC-AUC    : 0.3865
PR-AUC     : 0.4324
FPR        : 0.0000
FNR        : 1.0000
------------------------------------------------------------
Confusion Matrix:
              Predicted BENIGN   Predicted DDoS
Actual BENIGN        70,668                 0
Actual DDoS          55,717                 0
------------------------------------------------------------
Prediction time  : 0.4575 sec


In [5]:
#Quick verification — add Cell 5:
print("Value counts for Init_Win_bytes_forward in aligned UNSW-NB15:")
print(X_unsw_clean["Init_Win_bytes_forward"].describe())
print()
print("Value counts for Init_Win_bytes_backward in aligned UNSW-NB15:")
print(X_unsw_clean["Init_Win_bytes_backward"].describe())

Value counts for Init_Win_bytes_forward in aligned UNSW-NB15:
count    126385.000000
mean        231.998702
std          73.038532
min           0.000000
25%         255.000000
50%         255.000000
75%         255.000000
max         255.000000
Name: Init_Win_bytes_forward, dtype: float64

Value counts for Init_Win_bytes_backward in aligned UNSW-NB15:
count    126385.000000
mean        230.904680
std          74.580647
min           0.000000
25%         255.000000
50%         255.000000
75%         255.000000
max         255.000000
Name: Init_Win_bytes_backward, dtype: float64


In [10]:
import os
import json

cross_dataset_summary_path = os.path.join(cfg["paths"]["reports_dir"], "cross_dataset_all_results.json")

if os.path.exists(cross_dataset_summary_path):
    with open(cross_dataset_summary_path) as f:
        all_cross_results = json.load(f)
else:
    all_cross_results = {}

all_cross_results["unsw_nb15"] = {
    "dataset_name": "UNSW-NB15",
    "feature_overlap": "7/20 (35%)",
    "same_tool": False,
    "tool_used": "Argus + Bro-IDS",
    "n_samples": int(len(y_unsw_clean)),
    "accuracy": float(cross_metrics_unsw["accuracy"]),
    "precision": float(cross_metrics_unsw["precision"]),
    "recall": float(cross_metrics_unsw["recall"]),
    "f1": float(cross_metrics_unsw["f1"]),
    "roc_auc": float(cross_metrics_unsw["roc_auc"]),
    "pr_auc": float(cross_metrics_unsw["pr_auc"]),
    "fpr": float(cross_metrics_unsw["fpr"]),
    "fnr": float(cross_metrics_unsw["fnr"]),
    "tn": int(cross_metrics_unsw["tn"]),
    "fp": int(cross_metrics_unsw["fp"]),
    "fn": int(cross_metrics_unsw["fn"]),
    "tp": int(cross_metrics_unsw["tp"]),
    "notes": "Init_Win_bytes_forward/backward found to be semantically mismatched (capped 0-255 vs full 16-bit range in source domain)",
}

with open(cross_dataset_summary_path, "w") as f:
    json.dump(all_cross_results, f, indent=2)

print("Saved UNSW-NB15 results to:", cross_dataset_summary_path)

Saved UNSW-NB15 results to: d:\MSc_Project\Network_IDS_Framework\results/reports\cross_dataset_all_results.json


In [11]:
from src.cross_dataset.domain_shift import compute_domain_shift_report, print_domain_shift_summary

# Compare the 7 common features - BENIGN class first
X_2017_benign_common = X_train_common.loc[y_train == 0]
X_unsw_benign = X_unsw_clean.loc[y_unsw_clean == 0]

shift_benign_unsw = compute_domain_shift_report(X_2017_benign_common, X_unsw_benign)
print("BENIGN class domain shift (2017 vs UNSW-NB15, 7 common features):")
print_domain_shift_summary(shift_benign_unsw)

# Attack class
X_2017_attack_common = X_train_common.loc[y_train == 1]
X_unsw_attack = X_unsw_clean.loc[y_unsw_clean == 1]

shift_attack_unsw = compute_domain_shift_report(X_2017_attack_common, X_unsw_attack)
print("\nAttack class domain shift (2017 vs UNSW-NB15, 7 common features):")
print_domain_shift_summary(shift_attack_unsw)

# Save
shift_attack_unsw.to_csv(os.path.join(cfg["paths"]["reports_dir"], "domain_shift_unsw_nb15.csv"), index=False)
print("\nSaved domain_shift_unsw_nb15.csv")

BENIGN class domain shift (2017 vs UNSW-NB15, 7 common features):
DOMAIN SHIFT SUMMARY
Features tested:                7
Features with significant shift (KS p<0.05): 7/7
----------------------------------------------------------------
Top 10 most shifted features (by KS statistic):
                    feature  ks_statistic  wasserstein_normalized  significant_shift
          Total Fwd Packets      0.711671                1.316839               True
Total Length of Fwd Packets      0.637851                0.767611               True
     Fwd Packet Length Mean      0.636592                0.409517               True
    Init_Win_bytes_backward      0.602818                0.194550               True
     Init_Win_bytes_forward      0.565135                0.414852               True
               Bwd IAT Mean      0.449348                0.305817               True
               Fwd IAT Mean      0.387805                0.321672               True

Attack class domain shift (2017 vs U